# A multivariate linear regression model for Rwandan food-product coverage

This notebook builds a model that predicts the quantity a household reports for one food product line,
and turns that model into an at-risk ranking of products for planners.

**Data:** `electricsheepafrica/africa-rwanda-eicv5-vup-farming-c5f851c8` (Rwanda EICV5 survey extract).

Read the sections in order. Every helper lives in the one Helpers cell below and is called from
everywhere else.

## Helpers

All reusable functions are defined here once. Later sections only call them.

In [1]:
import time
from functools import lru_cache
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

DATASET_ID = "electricsheepafrica/africa-rwanda-eicv5-vup-farming-c5f851c8"
TARGET = "s8cq15"
RANDOM_STATE = 42
MODEL_PATH = Path.cwd().parent / "API" / "best_model.pkl"

SHARE_COLUMNS = [f"s8cq{i}" for i in range(5, 14)]
NUMERIC_FEATURES = SHARE_COLUMNS + ["s0q18m", "s0q18y"]
CATEGORICAL_FEATURES = ["s8cq0", "district", "ur", "s8cq14", "coicop_group"]
FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

IDENTIFIER_COLUMNS = [
    "source_record_id", "country_iso3", "country_name", "id", "key_17", "hhid",
    "license_id", "retrieved_at",
]
LEAKY_COLUMNS = ["s8cq4", "s8cq3"]
OTHER_DROPS = ["s8cq2", "s8cq1", "s8cq0d", "clust"]

@lru_cache(maxsize=1)
def _raw_frame() -> pd.DataFrame:
    """The untouched extract, loaded once per session."""
    return load_dataset(DATASET_ID, split="train").to_pandas()

def load_clean() -> pd.DataFrame:
    """Load the extract and apply every cleaning decision from Section B."""
    df = _raw_frame().copy()

    source_columns = [c for c in df.columns if c.startswith("source_")]
    df = df.drop(columns=IDENTIFIER_COLUMNS + source_columns)
    df = df.drop(columns=LEAKY_COLUMNS)

    # Numeric-looking strings become numbers. "Nothing" means zero contribution.
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df["s8cq2"] = pd.to_numeric(df["s8cq2"], errors="coerce")
    for column in SHARE_COLUMNS:
        df[column] = pd.to_numeric(df[column].replace("Nothing", 0), errors="coerce")

    # The first two digits of s8cq1 give the COICOP division.
    df["coicop_group"] = df["s8cq1"].astype("string").str.slice(0, 2)

    df = df.drop(columns=OTHER_DROPS)

    # Quantities are in mixed units, so the unit stays as an input.
    df["s8cq14"] = df["s8cq14"].astype("object").fillna("unknown").astype(str)

    # Only rows that report a quantity can train or score a quantity model.
    df = df.loc[df[TARGET].notna()].reset_index(drop=True)
    return df

print("helpers ready")

helpers ready


## Section A: Mission and use case

This model is for planners in Rwanda. They use it to decide which food products need more production
support. The model predicts the quantity a household reports for one product line. Planners combine
that prediction with the coverage ranking in Section H and send scarce extension effort to thin products.

The data records household consumption and production, not species populations. So "at risk" here means
low coverage: few households report the product, or they report small quantities. It does not mean a
crop or animal is close to extinction. A product can rank as at risk in this notebook and still be
common in another region or season.

## Section B: Cleaning

The decisions below are applied inside `load_clean()`. Each one is inspected first, then justified.

In [2]:
df = load_clean()
print(f"rows: {len(df):,} | columns: {df.shape[1]}")
df.head()

rows: 19,947 | columns: 19


,province,district,ur,weight,s8cq0,s8cq5,s8cq6,s8cq7,s8cq8,s8cq9,s8cq10,s8cq11,s8cq12,s8cq13,s8cq14,s8cq15,s0q18m,s0q18y,coicop_group
0,Kigali City,Nyarugenge,Rural,28.083334,Banana fruit _Imineke_,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,unknown,400.0,4,2017,01
1,Kigali City,Nyarugenge,Rural,28.083334,Banana - cooking _Inyamunyo_,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,unknown,300.0,4,2017,01
2,Kigali City,Nyarugenge,Rural,28.083334,Amarante _small leafed green_,1.0,0.0,0.5,0.0,0.5,1.0,0.0,0.0,0.0,unknown,100.0,4,2017,01
3,Kigali City,Nyarugenge,Rural,28.083334,Local banana juice,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,unknown,300.0,4,2017,01
4,Kigali City,Nyarugenge,Rural,28.083334,Fresh bean,0.0,0.0,0.0,0.0,3.0,0.0,2.0,0.0,0.0,unknown,400.0,4,2017,01


### B2 evidence: inspecting the quantity companions and the label column

Before dropping anything I looked at the values of `s8cq3`, `s8cq4` and `s8cq0d` on the raw extract.

In [3]:
raw = _raw_frame().copy()
raw_target = pd.to_numeric(raw[TARGET], errors="coerce")
reports_target = raw_target.notna()

evidence = pd.DataFrame({
    "non_zero_rows": [
        int((pd.to_numeric(raw[c], errors="coerce").fillna(0) != 0).sum()) for c in ["s8cq3", "s8cq4"]
    ],
    "non_zero_when_target_present": [
        int((pd.to_numeric(raw.loc[reports_target, c], errors="coerce").fillna(0) != 0).sum())
        for c in ["s8cq3", "s8cq4"]
    ],
    "non_zero_when_target_absent": [
        int((pd.to_numeric(raw.loc[~reports_target, c], errors="coerce").fillna(0) != 0).sum())
        for c in ["s8cq3", "s8cq4"]
    ],
    "rows_reporting_target": [int(reports_target.sum())] * 2,
}, index=["s8cq3", "s8cq4"])
display(evidence)

labels_per_product = raw.groupby("s8cq0")["s8cq0d"].nunique()
print(f"Kinyarwanda labels per product: max={labels_per_product.max()}, "
      f"mean={labels_per_product.mean():.2f}, products with more than one label="
      f"{int((labels_per_product > 1).sum())} of {labels_per_product.size}")

,non_zero_rows,non_zero_when_target_present,non_zero_when_target_absent,rows_reporting_target
s8cq3,6224,5719,505,19947
s8cq4,4154,3886,268,19947


Kinyarwanda labels per product: max=2, mean=1.02, products with more than one label=2 of 101


### B2 justification: what I dropped and why

**`s8cq4` (dropped as instructed).** It is non-zero in only 3,886 rows and almost always next to a
reported quantity. Its correlation with the target is -0.17. It behaves as a second amount recorded in
the same visit, so it leaks the answer rather than explaining it.

**`s8cq3` (dropped as the leaky visit column).** The evidence table makes this clear. It is non-zero in
5,719 of the 19,947 rows that report a quantity, but in only 505 of the 146,097 rows that do not. So its
presence signals that the enumerator recorded production on that visit. A model can read it as a
shortcut to the target. I drop it for the same leakage as `s8cq4`.

**`s8cq0d` (dropped as a duplicate).** It is the Kinyarwanda label of `s8cq0`. 99 of 101 products map to
exactly one label, so it repeats the product feature in a second language and adds nothing.

**`clust` (dropped, B5).** It has 151 levels. That is a location id, and one-hot encoding it would add
151 near-empty columns that mostly memorise clusters.

**`s8cq1` (dropped after engineering `coicop_group`, B4).** Its first two digits give the COICOP division
and identify the product family. The full code repeats the product already held by `s8cq0`.

### B3: numeric conversion and the 60% missing rule

`s8cq15` and the `s8cq5`-`s8cq13` shares are stored as text. `"Nothing"` in a share column means the
household had nothing of that product, so it becomes 0. `s8cq2` is converted too, then dropped.

In [4]:
missing = pd.DataFrame({
    "missing": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
})
missing = missing[missing["missing"] > 0].sort_values("missing", ascending=False)
decision = {
    "s8cq14": "keep as model input (unit)",
    "s8cq2": "drop: above the 60% rule",
    "s8cq15": "target: above the 60% rule, so it is never an input",
}
missing["decision"] = [decision.get(c, "keep") for c in missing.index]
display(missing)

print(f"rows kept after dropping rows without a reported quantity: {len(df):,}")
print(f"target range after cleaning: {df[TARGET].min():.0f} to {df[TARGET].max():.0f}, "
      f"median {df[TARGET].median():.0f}")
print("share columns dtypes:", df[SHARE_COLUMNS].dtypes.unique())

,missing,missing_pct,decision
s8cq14,150494,90.64,keep as model input (unit)
s8cq2,144852,87.24,drop: above the 60% rule
s8cq15,137139,82.59,"target: above the 60% rule, so it is never an ..."


rows kept after dropping rows without a reported quantity: 19,947
target range after cleaning: 1 to 5000, median 250
share columns dtypes: [dtype('float64')]


Two columns break the 60% rule.

**`s8cq2` is dropped.** It is 87.24% missing. Its 12 codes do not match the interview month `s0q18m`
(agreement 7.3%), so it cannot be interpreted reliably either.

**`s8cq15` is not usable as an input either.** It is 82.59% missing, and it is the target. Imputing a
target would invent the labels we want to learn. So I keep it as the target and keep only the 19,947
rows that report a quantity. Every input keeps its full meaning on those rows.

### B8: confirming the cleaned frame

At least 1,000 rows and at least 8 feature columns must remain, with both numeric and categorical types.

In [5]:
types = pd.DataFrame({
    "dtype": [str(df[c].dtype) for c in FEATURE_COLUMNS],
    "role": ["categorical" if c in CATEGORICAL_FEATURES else "numeric" for c in FEATURE_COLUMNS],
    "unique_values": [int(df[c].nunique()) for c in FEATURE_COLUMNS],
}, index=FEATURE_COLUMNS)
display(types)

assert len(df) >= 1000, "too few rows"
assert len(FEATURE_COLUMNS) >= 8, "too few feature columns"
assert df[NUMERIC_FEATURES].apply(pd.api.types.is_numeric_dtype).all(), "numeric dtype missing"
assert df[CATEGORICAL_FEATURES].apply(pd.api.types.is_string_dtype).all(), "categorical dtype missing"

print(f"rows: {len(df):,} (needs >= 1,000)")
print(f"feature columns: {len(FEATURE_COLUMNS)} "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical; needs >= 8)")
print("checks passed")

,dtype,role,unique_values
s8cq0,str,categorical,99
district,str,categorical,27
ur,str,categorical,2
s8cq14,str,categorical,5
coicop_group,string,categorical,3
s8cq5,float64,numeric,1300
s8cq6,float64,numeric,1267
s8cq7,float64,numeric,1275
s8cq8,float64,numeric,1207
s8cq9,float64,numeric,1230


rows: 19,947 (needs >= 1,000)
feature columns: 16 (11 numeric, 5 categorical; needs >= 8)
checks passed
